
# Maximum-entropy FRET distance recovery, and picking nu

Recover a FRET distance distribution ``p(R_DA)`` from a simulated TCSPC
decay with ``solve_tcspc_mem_fret``, and compare three ways of choosing the
entropy weight nu: deliberately under-regularised, deliberately
over-regularised, and the opt-in "historic MaxEnt" mode that finds nu
automatically by targeting a reduced chi-square of 1.

The lesson the figure encodes, visible in the panels: the *wrong*
regularisation fits the data better than the truth does. At tiny nu the
distribution collapses into sharp spikes and chi-square drops *below* 1 --
that is not a better answer, it is the classic maximum-entropy failure mode.
At large nu the fit is pulled toward the flat prior and smeared. The
auto-nu search sits between the two and recovers both the centre and the
width of the true distribution.

Everything here is simulated: one million photons from ``SimEngine``, a
Gaussian distance distribution (mean 45 A, sigma 4 A) as ground truth.

<div class="alert alert-info"><h4>Note</h4><p>This example is the generator of ``doc/img/maxent_fret_distance_recovery.png``
   used in the documentation, with fixed seeds so the figure is reproducible.</p></div>


In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

import tttrlib


# Ground truth: a Gaussian distance distribution turned into a lifetime
# mixture via the Foerster rate at each distance.
TAU0, R0 = 4.0, 50.0
R_MEAN, R_SIGMA = 45.0, 4.0
N_BINS, PERIOD = 1024, 25.6
DT = PERIOD / N_BINS

R_sim = np.linspace(28.0, 72.0, 45)
w_sim = np.exp(-0.5 * ((R_sim - R_MEAN) / R_SIGMA) ** 2)
w_sim /= w_sim.sum()
tau_sim = 1.0 / (1.0 / TAU0 + (1.0 / TAU0) * (R0 / R_sim) ** 6)

t = np.arange(N_BINS) * DT
irf = np.exp(-0.5 * ((t - 2.0) / 0.3) ** 2)
irf /= irf.sum()

# Simulate. Background is zeroed: the simulator's unconfigured background is
# a delta spike at micro-time 0, not a flat floor, and would bias any
# distance recovery.
cfg = json.loads(tttrlib.SimEngine.default_json())
cfg["settings"].update(
    n_ph_max=1_000_000, max_windows=10 ** 8,
    n_microtime_channels=N_BINS, microtime_resolution=DT,
    laser_period=PERIOD, seed_diffusion=7, seed_emission=8)
cfg["background"] = [0.0, 0.0]
cfg["species"][0]["decay"] = {
    "lifetimes": tau_sim.tolist(), "amplitudes": w_sim.tolist(),
    "dt": DT, "n_bins": N_BINS, "irf": irf.tolist()}
engine = tttrlib.SimEngine.from_dict(cfg)
engine.run()
hist = np.bincount(
    np.asarray(engine.photons()["micro_time"]),
    minlength=N_BINS).astype(float)[:N_BINS]
n_photons = hist.sum()
print(f"simulated photons: {n_photons:.0f}")

R_grid = np.arange(25.0, 75.01, 0.5)
dR = R_grid[1] - R_grid[0]


def solve(nu=1e-5, target=-1.0):
    """One MEM solve on the shared histogram. ``target > 0`` switches on the
    auto-nu search; the ``nu`` argument then only seeds it."""
    return tttrlib.solve_tcspc_mem_fret(
        hist.tolist(), irf.tolist(), DT, R_grid.tolist(), TAU0, R0,
        [1.0, TAU0], 0.0, 0.0, 0.0, 0.0, 5, N_BINS - 1, 0.0,
        irf_background=0.0, nu=nu, max_iter=500, target_chisq=target)


res_auto = solve(target=1.0)
res_under = solve(nu=1e-9)
res_over = solve(nu=3e-4)
print(f"auto-nu: converged={res_auto.target_chisq_converged} "
      f"nu={res_auto.nu_used:.3e} chisq={res_auto.chisq:.4f}")


def density(p):
    p = np.asarray(p, float)
    total = p.sum()
    return p / (total * dR) if total > 0 else p


def stats(p):
    """Mean and standard deviation of the recovered distribution."""
    p = np.asarray(p, float)
    p = p / p.sum()
    mean = float((p * R_grid).sum())
    sd = float(np.sqrt((p * (R_grid - mean) ** 2).sum()))
    return mean, sd


truth = np.exp(-0.5 * ((R_grid - R_MEAN) / R_SIGMA) ** 2)
truth /= truth.sum() * dR

methods = [
    ("unregularised (ESM)", density(res_under.p_esm),
     stats(res_under.p_esm), res_under.chisq_esm, "tab:red"),
    (r"fixed $\nu = 10^{-9}$ (under)", density(res_under.p),
     stats(res_under.p), res_under.chisq, "tab:orange"),
    (r"fixed $\nu = 3\times10^{-4}$ (over)", density(res_over.p),
     stats(res_over.p), res_over.chisq, "tab:green"),
    (rf"auto-$\nu$, target $\chi^2_r=1$ ($\nu$={res_auto.nu_used:.1e})",
     density(res_auto.p), stats(res_auto.p), res_auto.chisq, "tab:blue"),
]

fig = plt.figure(figsize=(12, 6.4))
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.95],
                      hspace=0.42, wspace=0.28,
                      left=0.06, right=0.98, top=0.88, bottom=0.09)
axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]
ax_nu = fig.add_subplot(gs[:, 2])

ymax = 1.9 * truth.max()
for ax, (title, dens, (mean, sd), chi2, color) in zip(axes, methods):
    ax.fill_between(R_grid, truth, color="0.86", zorder=0)
    ax.plot(R_grid, np.minimum(dens, ymax * 1.05), color=color, lw=1.9)
    ax.set_ylim(0, ymax)
    ax.set_xlim(25, 75)
    ax.set_title(title, fontsize=10)
    peak = dens.max()
    if peak > ymax:  # the under-regularised spikes run off-scale: say by how much
        i_peak = int(np.argmax(dens))
        ax.annotate(f"peak {peak:.2f} \u2191", (R_grid[i_peak], ymax * 0.93),
                    fontsize=8, ha="center", color=color)
    ax.text(0.02, 0.95,
            f"$\\chi^2_r$ = {chi2:.2f}\n"
            f"$\\bar{{R}}$ = {mean:.1f} \u00c5   $\\sigma$ = {sd:.1f} \u00c5",
            transform=ax.transAxes, va="top", fontsize=9)
for ax in axes[2:]:
    ax.set_xlabel(r"$R_{DA}$ ($\mathrm{\AA}$)")
for ax in (axes[0], axes[2]):
    ax.set_ylabel(r"$p(R)$ ($\mathrm{\AA}^{-1}$)")

# The mechanism panel: chi-square is monotone in nu, and the auto-nu search
# lands where it crosses the target.
nus = np.geomspace(1e-10, 1e-1, 15)
chis = [solve(nu=float(nu)).chisq for nu in nus]
ax_nu.semilogx(nus, chis, "o-", ms=3.5, color="0.45")
ax_nu.axhline(1.0, color="tab:blue", ls="--", lw=1)
ax_nu.axvline(res_auto.nu_used, color="tab:blue", ls=":", lw=1)
ax_nu.plot([res_auto.nu_used], [res_auto.chisq], "o", color="tab:blue", ms=8)
ax_nu.annotate(f"found $\\nu$ = {res_auto.nu_used:.1e}\n"
               f"$\\chi^2_r$ = {res_auto.chisq:.2f}",
               (res_auto.nu_used, res_auto.chisq),
               textcoords="offset points", xytext=(10, 18), fontsize=9,
               color="tab:blue")
ax_nu.set_yscale("log")
ax_nu.set_xlabel(r"$\nu$")
ax_nu.set_ylabel(r"$\chi^2_r$")
ax_nu.set_title("auto-$\\nu$ search: the controller lands\n"
                "$\\chi^2_r = 1$ on the monotone $\\chi^2_r(\\nu)$ curve",
                fontsize=10)

fig.suptitle(
    "FRET distance-distribution recovery by maximum entropy \u2014 "
    f"truth (grey, Gaussian $\\bar{{R}}$ = {R_MEAN:.0f} \u00c5, "
    f"$\\sigma$ = {R_SIGMA:.0f} \u00c5) vs recovered\n"
    f"photons simulated by SimEngine (N = {n_photons:.0f}, "
    f"$\\tau_0$ = {TAU0} ns, $R_0$ = {R0} \u00c5), "
    "recovered with solve_tcspc_mem_fret",
    fontsize=11)
plt.show()